In [ ]:
# To import dataset from google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# For the pretrained weights - needed for importing torchtext
!pip install torch==2.3.0 torchtext==0.18.0

import torchtext
print(torchtext.__version__)

In [ ]:
# Google colab notebook used for the experiments with pretrained weights

# Imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import re
import torch.nn.functional as F
from functools import partial
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np
import os
import torchtext
from torchtext.vocab import GloVe

# =========================================================
# 1. LOAD SST5 DATASET
# =========================================================
ds = load_dataset("SetFit/sst5")

# Basic English Tokenizer
def tokenizer(text):
    text = text.lower()
    tokens = re.findall(r"\b[\w']+\b", text)
    return tokens

# Vocabulary Class
class Vocab:
    def __init__(self, specials=["<pad>", "<unk>"]):
        self.itos = list(specials)
        self.stoi = {tok: idx for idx, tok in enumerate(self.itos)}

    # Builds vocabulary from a list of texts.
    def build_vocab(self, texts, min_freq=1):
        freq = {}
        for text in texts:
            for token in tokenizer(text):
                freq[token] = freq.get(token, 0) + 1
        for token, count in freq.items():
            if count >= min_freq:
                self.stoi[token] = len(self.itos)
                self.itos.append(token)

    def __call__(self, tokens):
        return [self.stoi.get(tok, self.stoi["<unk>"]) for tok in tokens]

# Dataset Classes
class SST5Dataset(Dataset):
    def __init__(self, dataset, vocab):
        self.dataset = dataset
        self.vocab = vocab

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]['text']
        label = self.dataset[idx]['label']
        text_ids = torch.tensor(self.vocab(tokenizer(text)), dtype=torch.long)
        label = torch.tensor(label, dtype=torch.long)
        return text_ids, label

# Collate Functions
def collate_batch(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    labels = torch.stack(labels)
    return texts_padded, labels

def collate_batch_mlp(batch, vocab_size):
    texts, labels = zip(*batch)
    labels = torch.stack(labels)

    batch_bow = []
    for text_ids in texts:
        bow_vec = torch.bincount(text_ids, minlength=vocab_size)
        batch_bow.append(bow_vec)

    texts_bow = torch.stack(batch_bow).float()

    return texts_bow, labels

# Build vocabulary (min_freq=1 because the dataset is small)
vocab = Vocab()
vocab.build_vocab([example['text'] for example in ds['train']])

# Model hyperparameters
VOCAB_SIZE = len(vocab.itos)
EMB_DIM = 100
HIDDEN = 100
NUM_CLASSES = 5
PAD_IDX = 0
UNK_IDX = 1
MAX_SEQ_LEN_SST5 = 1000

collate_fn_mlp = partial(collate_batch_mlp, vocab_size=VOCAB_SIZE)

# Load Pre-trained GloVe Embeddings
print(f"Loading GloVe embeddings (dim={EMB_DIM})...")
glove_vectors = GloVe(name='6B', dim=EMB_DIM)

# Create the pre-trained weight matrix
weights_matrix = torch.zeros((VOCAB_SIZE, EMB_DIM))
words_found = 0

for i, word in enumerate(vocab.itos):
    try:
        weights_matrix[i] = glove_vectors.get_vecs_by_tokens(word)
        words_found += 1
    except KeyError:
        weights_matrix[i] = torch.tensor(np.random.normal(scale=0.6, size=(EMB_DIM, )))

device = "cuda" if torch.cuda.is_available() else "cpu"
# Convert to a tensor and move to device
weights_matrix = weights_matrix.to(device)

# =========================================================
# 2. CREATE DATASETS
# =========================================================
train_dataset = SST5Dataset(ds['train'], vocab)
val_dataset = SST5Dataset(ds['validation'], vocab)
test_dataset = SST5Dataset(ds['test'], vocab)

# =========================================================
# 3. DATALOADERS
# =========================================================
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_batch)

# For MLP (BoW)
train_loader_mlp = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn_mlp)
val_loader_mlp = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn_mlp)
test_loader_mlp = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn_mlp)

class MLP(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(vocab_size, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # x shape: (batch_size, vocab_size)
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, num_classes, pad_idx, filter_sizes=[3, 4, 5], num_filters=100, dropout=0.5):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        # Convolution layers for each filter size
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=emb_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])

        # Fully connected classification layer
        self.fc = nn.Linear(len(filter_sizes) * num_filters, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)
        emb = emb.permute(0, 2, 1)

        # Apply convolution + ReLU
        conv_outputs = [F.relu(conv(emb)) for conv in self.convs]

        # Global max pooling for each feature map
        pooled_outputs = [F.max_pool1d(conv_out, conv_out.shape[2]).squeeze(2)
                          for conv_out in conv_outputs]

        # Concatenate all pooled features
        cat = torch.cat(pooled_outputs, dim=1)
        cat = self.dropout(cat)
        out = self.fc(cat)

        return out

# GRU
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_classes, pad_idx):
        super().__init__()
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        # GRU layer
        self.gru = nn.GRU(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            bidirectional=False,
            batch_first=True
        )

        # Classification layer
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, emb_dim)
        _, hidden = self.gru(embedded)

        # Use last hidden state → (batch_size, hidden_dim)
        last_hidden = hidden[-1, :, :]

        out = self.dropout(last_hidden)
        out = self.fc(out)
        return out

class PositionalEncoding(nn.Module):
    def __init__(self, emb_dim, max_len=1000):
        super().__init__()

        pe = torch.zeros(max_len, emb_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, emb_dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / emb_dim))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0)) # (1, max_len, emb_dim)

    def forward(self, x):
        # x is (batch_size, seq_len, emb_dim)
        x = x + self.pe[:, :x.size(1), :]
        return x

class TransformerEncoderClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim,
        hidden_dim,
        num_classes,
        pad_idx,
        max_seq_len=1000,
        num_layers=2,
        nhead=4,
        dropout=0.5
    ):
        super().__init__()

        ff_dim = hidden_dim * 4

        # Embedding + positional encoding
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(emb_dim, max_len = max_seq_len)
        self.dropout = nn.Dropout(dropout)

        # Transformer Encoder Stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head
        self.fc = nn.Linear(emb_dim, num_classes)
        self.pad_idx = pad_idx

    def forward(self, x):
        # x: (batch_size, seq_len)

        # Embedding & positional encoding
        embedded = self.embedding(x)  # (batch_size, seq_len, emb_dim)
        embedded = self.pos_encoder(embedded)
        embedded = self.dropout(embedded)

        # Mask for padding tokens
        src_key_padding_mask = (x == self.pad_idx)

        # Transformer encoder output
        output = self.transformer_encoder(
            embedded,
            src_key_padding_mask=src_key_padding_mask
        )

        # Sequence representation: mean pooling
        cls_output = output.mean(dim=1) # (batch_size, emb_dim)

        out = self.fc(cls_output)
        return out

# =========================================================
# 4. MODEL DEFINITIONS
# =========================================================
model_cnn = CNNClassifier(
    vocab_size=VOCAB_SIZE,
    emb_dim=EMB_DIM,
    num_classes=NUM_CLASSES,
    pad_idx=PAD_IDX
).to(device)
model_cnn.embedding.weight.data.copy_(weights_matrix)

model_gru = GRUClassifier(
    vocab_size=VOCAB_SIZE,
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN,
    num_classes=NUM_CLASSES,
    pad_idx=PAD_IDX
).to(device)
model_gru.embedding.weight.data.copy_(weights_matrix)

model_transformer = TransformerEncoderClassifier(
    vocab_size=VOCAB_SIZE,
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN,
    num_classes=NUM_CLASSES,
    pad_idx=PAD_IDX
).to(device)
model_transformer.embedding.weight.data.copy_(weights_matrix)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_model(model, loader, criterion, device):
    model.eval()
    correct, total = 0, 0
    total_loss = 0.0
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)

            # Calculate validation loss
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Calculate validation accuracy
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    # Return average accuracy AND average loss
    return correct / total, total_loss / len(loader)

# =========================================================
# 5. OUTPUT DIRECTORIES
# =========================================================
MODELS = "/content/drive/MyDrive/sst5_models_weights_final"
PLOTS = "/content/drive/MyDrive/sst5_plots_weights_final"
os.makedirs(MODELS, exist_ok=True)
os.makedirs(PLOTS, exist_ok=True)

# =========================================================
# 6. UNIFIED TRAINING FUNCTION
# =========================================================
def run_experiment(model_name, model, train_loader, valid_loader, epochs=10, lr=0.001):
    print(f"\n--- Running {model_name} Experiment ---")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_valid_acc = -1.0
    best_model_path = os.path.join(MODELS, f"{model_name.replace(' ', '_')}_best_model.pt")

    # Dictionary to store training progress history
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_acc': []
    }

    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_acc, valid_loss = eval_model(model, valid_loader, criterion, device)

        print(f"Epoch {epoch+1}/{epochs}: Train loss={train_loss:.4f}, Valid loss={valid_loss:.4f}, Valid acc={valid_acc:.4f}")

        # Logging values to history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(valid_loss)
        history['val_acc'].append(valid_acc)

        # Model Saving Logic
        if valid_acc > best_valid_acc:
            best_valid_acc = valid_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  -> New best model saved to {best_model_path} (Acc: {valid_acc:.4f})")

    print(f"Training complete. Best model saved at {best_model_path}")
    # Return the path AND the history
    return best_model_path, history

# Collect Predictions from Classification Model
def get_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)

            outputs = model(texts)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return all_preds, all_labels

# Print Classification Metrics
def print_classification_report(labels, preds, class_names):
    acc = accuracy_score(labels, preds)

    # Generates P, R, F1, and support for each class
    report = classification_report(labels, preds, target_names=class_names)

    print("="*30)
    print(f"FINAL TEST METRICS")
    print("="*30)
    print(f"Overall Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(report)
    print("="*30)

# Plots training/validation loss and validation accuracy over epochs.
def plot_training_curves(history, model_name, plots_file):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss Curve
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Validation Loss')
    ax1.set_title(f"{model_name} - Loss Curves")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()

    # Accuracy Curve
    ax2.plot(history['val_acc'], label='Validation Accuracy', color='green')
    ax2.set_title(f"{model_name} - Validation Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()

    plt.tight_layout()

    # Save figure into plots_file
    filename = os.path.join(plots_file, f"{model_name.replace(' ', '_')}_training_curves.png")
    plt.savefig(filename)
    print(f"Plot saved: {filename}")
    plt.close(fig)

# Plots confusion matrix heatmap and saves the figure.
def plot_confusion_matrix(labels, preds, model_name, plots_file, class_names):
    cm = confusion_matrix(labels, preds)

    fig = plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")

    # Save figure into plots_file
    filename = os.path.join(plots_file, f"{model_name.replace(' ', '_')}_confusion_matrix.png")
    plt.savefig(filename)
    print(f"Plot saved: {filename}")
    plt.close(fig)

# Final Test Evaluation Using Best Model Weights
def test_final_model(model_name, model, best_model_path, test_loader, device, plots_file, cls_num):
    print(f"\n--- Loading best '{model_name}' for FINAL TEST ---")

    # Load best weights
    try:
        model.load_state_dict(torch.load(best_model_path))
    except FileNotFoundError:
        print(f"ERROR: Best model path not found at {best_model_path}.")

    # Predictions
    all_preds, all_labels = get_predictions(model, test_loader, device)

    # Class labels
    if cls_num == 2:
        cls = ['Negative', 'Positive']
    else:
        cls = ['V. Negative', 'Negative', 'Neutral', 'Positive', 'V. Positive']

    # # Metrics (Accuracy, P, R, F1)
    print_classification_report(all_labels, all_preds, cls)

    # Confusion Matrix
    plot_confusion_matrix(all_labels, all_preds, model_name, plots_file, cls)

# =========================================================
# 7. TRAIN ALL MODELS
# =========================================================
#CNN
cnn_path, cnn_history = run_experiment("Kim_CNN", model_cnn,
    train_loader,
    val_loader,
    epochs=10
)
test_final_model("Kim_CNN", model_cnn, cnn_path, test_loader, device, PLOTS, NUM_CLASSES)
plot_training_curves(cnn_history, "Kim_CNN", PLOTS)

#GRU
gru_path, gru_history = run_experiment("GRU_weights", model_gru,
    train_loader,
    val_loader,
    epochs=10
)
test_final_model("GRU_weights", model_gru, gru_path, test_loader, device, PLOTS, NUM_CLASSES)
plot_training_curves(gru_history, "GRU_weights", PLOTS)

#Transformer
transformer_path, transformer_history = run_experiment("Transformer_weights",
    model_transformer,
    train_loader,
    val_loader,
    epochs=10,
    lr=0.0001
)
test_final_model("Transformer_weights", model_transformer, transformer_path, test_loader, device, PLOTS, NUM_CLASSES)
plot_training_curves(transformer_history, "Transformer_weights", PLOTS)